In [8]:
import os 
import json 
import re 
from enum import Enum
from openai import OpenAI
from pydantic import BaseModel

In [9]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPEN_AI_API")
try:
    client = OpenAI(api_key=api_key)
except ImportError:
    print("OpenAI library not found.")

#### **Umbrela Scoring model**

In [10]:
class UMBRELAScoreValues(str , Enum):
    NO_RELEVANCE = "0"
    RELATED = "1"
    PARTIAL_ANSWER = "2"
    EXACT_ANSWER = "3"

class UMBRELAScore(BaseModel):
    score : UMBRELAScoreValues

In [11]:
UMBRELA_PROMPT = """
Given a query and a passage, you must provide a score on an integer scale of 0 to 3 with the following meanings:
    0 = represent that the passage has nothing to do with the query,
    1 = represents that the passage seems related to the query but does not answer it,
    2 = represents that the passage has some answer for the query, but the answer may be a bit unclear, or hidden amongst extraneous information and
    3 = represents that the passage is dedicated to the query and contains the exact answer.
    
Important Instructions: 
Assign category 1 if the passage is somewhat related to the topic but not completely, 
category 2 if passage presents something very important related to the entire topic but also has some extra information and 
category 3 if the passage only and entirely refers to the topic. 
If none of the above satisfies give it category 0.

Query: {query}
Passage: {chunk}

Split this problem into steps:
Consider the underlying intent of the search.
Measure how well the content matches a likely intent of the query (M).
Measure how trustworthy the passage is (T).
Consider the aspects above and the relative importance of each, and decide on a final score (O). Final score must be an integer value only.
Do not provide any code in result. Provide each score in the format of: a single integer without any reasoning.
"""

#### **Implementation**

In [12]:
def compute_umbrela(query : str , chunks : list) -> dict[str , int]:
    """
        Iterate over retrieved chunks and score each one against the query 
        using the structured prompt.
        Args:
            query(str) : User question
            chunks(list) : Retrieved list 
        Returns:
            Dict[str , int]
    """
    scores = {}
    model_kwargs = {
        "seed" : 42,
        "reasoning_effort" : "minimal"
    }
    for chunk in chunks:
        try:
            prompt = UMBRELA_PROMPT.format(query = query , chunk = chunk)
            completion = client.beta.chat.completions.parse(
                model = "gpt-5-mini",
                messages = [
                    {
                        "role" : "system",
                        "content" : "You are a helpful assistant that follows user instructions precisely and provide accurate information."
                    },
                    {"role" : "user" , "content" : prompt}
                ] ,
                response_format=UMBRELAScore,
                **model_kwargs
            )

            response = completion.choices[0].message.parsed
            if not response.score:
                raise ValueError(f"Failed to parse response")
            
            scores[chunk] = int(response.score.value)

        except Exception as e:
            raise Exception(f"Error computing UMBRELA score : {str(e)}")
        
    return scores 

#### **Evaluation Examples**

In [13]:
query1 = "What were the main causes of the fall of the Roman Empire?"
chunks1 = [
    "The Roman Empire's fall is often attributed to a combination of economic instability, overexpansion, and military overspending. Constant wars and heavy taxation strained the treasury.",
    "Barbarian invasions from groups like the Goths and Vandals placed immense pressure on Roman borders, leading to the sack of Rome in 410 AD.",
    "The internal political climate was unstable, with a rapid succession of emperors and civil wars that weakened the state from within.",
    "The Great Wall of China was built to protect against nomadic invasions in the north, a significant engineering feat of its time."
]

# Example 2: Nuanced / Technical Topic
query2 = "How does photosynthesis work in plants?"
chunks2 = [
    "Photosynthesis is the process used by plants, algae, and certain bacteria to convert light energy into chemical energy, through a process that converts carbon dioxide and water into glucose and oxygen.",
    "Chlorophyll, the pigment that gives plants their green color, is crucial for absorbing sunlight in organelles called chloroplasts.",
    "Mitochondria are known as the powerhouses of the cell, responsible for generating most of the cell's supply of adenosine triphosphate (ATP)."
]

In [14]:
scores1 = compute_umbrela(query1, chunks1)
print(f"Query: '{query1}'")
print("UMBRELA retrieval Scores:")
for chunk, score in scores1.items():
    print(f"- Score {score} for chunk: '{chunk}...'")
print("-" * 45)

Query: 'What were the main causes of the fall of the Roman Empire?'
UMBRELA retrieval Scores:
- Score 3 for chunk: 'The Roman Empire's fall is often attributed to a combination of economic instability, overexpansion, and military overspending. Constant wars and heavy taxation strained the treasury....'
- Score 2 for chunk: 'Barbarian invasions from groups like the Goths and Vandals placed immense pressure on Roman borders, leading to the sack of Rome in 410 AD....'
- Score 2 for chunk: 'The internal political climate was unstable, with a rapid succession of emperors and civil wars that weakened the state from within....'
- Score 0 for chunk: 'The Great Wall of China was built to protect against nomadic invasions in the north, a significant engineering feat of its time....'
---------------------------------------------


In [15]:
scores2 = compute_umbrela(query2, chunks2)
print(f"Query: '{query2}'")
print("UMBRELA retrieval Scores:")
for chunk, score in scores2.items():
    print(f"- Score {score} for chunk: '{chunk}...'")
print("-" * 45)

Query: 'How does photosynthesis work in plants?'
UMBRELA retrieval Scores:
- Score 3 for chunk: 'Photosynthesis is the process used by plants, algae, and certain bacteria to convert light energy into chemical energy, through a process that converts carbon dioxide and water into glucose and oxygen....'
- Score 2 for chunk: 'Chlorophyll, the pigment that gives plants their green color, is crucial for absorbing sunlight in organelles called chloroplasts....'
- Score 0 for chunk: 'Mitochondria are known as the powerhouses of the cell, responsible for generating most of the cell's supply of adenosine triphosphate (ATP)....'
---------------------------------------------
